# Description
Run dictionary-based value narrative analysis for the core paper figures.

# Input
data/metadata/analysis_corpus_labeled.csv

# Output
results/value_timeseries_labeled.csv and related figures in figures/



# 02 Value Dictionary Analysis

## Description
Analyzes the frequency and co-occurrence of value vocabulary terms across the corpus.
Two complementary analyses are performed:
1. **Time series** – Monthly frequency of 8 value terms per 1,000 words, broken down by actor (State/Institution) and country
2. **Co-occurrence network** – Sentence-level co-occurrence of 10 value/conflict terms, visualized as a circular network graph

## Input
- `corpus/analysis_corpus_labeled.csv` – Labeled analysis corpus (output of `01_preprocessing.ipynb`)

## Output
- `outputs/value_timeseries_labeled.csv` – Monthly value term frequencies
- `outputs/fig_value_{term}_timeseries.png` – Time series plots (8 figures)
- `outputs/value_cooccur_edges.csv` – Co-occurrence edge list (top 30 pairs)
- `outputs/fig_value_cooccur_network.png` – Co-occurrence network visualization


In [ ]:
import os, re, math, itertools
from collections import Counter
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

INPUT   = "./corpus/analysis_corpus_labeled.csv"
OUT_DIR = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# Value dictionary (8 terms)
VALUES = {
    "sovereignty":           [r"\bsovereignty\b"],
    "territorial_integrity": [r"\bterritorial integrity\b"],
    "democracy":             [r"\bdemocracy\b", r"\bdemocratic\b"],
    "security":              [r"\bsecurity\b"],
    "civilization":          [r"\bcivilization\b", r"\bcivilisational\b"],
    "international_law":     [r"\binternational law\b"],
    "human_rights":          [r"\bhuman rights\b"],
    "rules_based":           [r"\brules[- ]based\b", r"\brules based\b"],
}

def load_text(p):
    try:
        with open(p, "r", encoding="utf-8") as f: return f.read().lower()
    except Exception: return ""

df = pd.read_csv(INPUT, dtype=str)
df = df[df["clean_path"].notna()].copy()
df["month"]   = df.get("month", df.get("date","")).fillna("").astype(str).str.slice(0,7)
df["text"]    = [load_text(p) for p in df["clean_path"]]
df["n_words"] = df["text"].str.split().apply(len).clip(lower=1)

for k, pats in VALUES.items():
    rx = re.compile("|".join(pats), flags=re.I)
    df[k] = df["text"].apply(lambda t: len(rx.findall(t))) / df["n_words"] * 1000

agg_cols = list(VALUES.keys())
g = df.groupby(["actor","country","month"])[agg_cols].mean().reset_index()
g.to_csv(f"{OUT_DIR}/value_timeseries_labeled.csv", index=False)
print(f"Saved value_timeseries_labeled.csv  rows={len(g)}")


In [ ]:
# Time series plots (one per value term)
TARGETS = [("State","Russia"),("State","Ukraine"),("State","Georgia"),("Institution","Institution")]
COLORS  = {"Russia":"firebrick","Ukraine":"royalblue","Georgia":"seagreen","Institution":"darkorange"}

for val in agg_cols:
    plt.figure(figsize=(13,5))
    for actor, country in TARGETS:
        sub = g[(g["actor"]==actor) & (g["country"]==country)].sort_values("month")
        if sub.empty: continue
        ls = "--" if actor == "Institution" else "-"
        plt.plot(sub["month"], sub[val], label=f"{actor}-{country}",
                 color=COLORS.get(country,"gray"), linestyle=ls, linewidth=1.4)
    plt.xticks(rotation=60, ha="right", fontsize=8)
    plt.title(f"Value term per 1000 words: {val}")
    plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/fig_value_{val}_timeseries.png", dpi=150); plt.close()
    print(f"Saved fig_value_{val}_timeseries.png")


In [ ]:
# Value co-occurrence network
VALUES_TERMS = ["sovereignty","territorial integrity","democracy","security",
                "civilization","aggression","occupation","sanction","rules-based","multipolar"]

co = Counter()
for t in df["text"].tolist():
    sents = re.split(r"[.!?]\s+", t)
    for s in sents:
        hits = sorted(set(term for term in VALUES_TERMS if term in s))
        for a, b in itertools.combinations(hits, 2):
            co[(a,b)] += 1

top = co.most_common(30)
edges = [(a,b,w) for (a,b),w in top]
pd.DataFrame(edges, columns=["term_a","term_b","count"]).to_csv(f"{OUT_DIR}/value_cooccur_edges.csv", index=False)

nodes = sorted(set([a for a,_,_ in edges]+[b for _,b,_ in edges]))
n = len(nodes)
pos = {nodes[i]: (math.cos(2*math.pi*i/n), math.sin(2*math.pi*i/n)) for i in range(n)}

fig, ax = plt.subplots(figsize=(10,10))
max_w = max(w for _,_,w in edges) if edges else 1
for a,b,w in edges:
    xa,ya=pos[a]; xb,yb=pos[b]
    ax.plot([xa,xb],[ya,yb], linewidth=0.5+4*w/max_w, alpha=0.6, color="steelblue")
for node,(x,y) in pos.items():
    ax.scatter([x],[y], s=80, zorder=5)
    ax.text(x*1.12, y*1.12, node, fontsize=9, ha="center", va="center")
ax.axis("off"); ax.set_title("Value Vocabulary Co-occurrence (Top 30 pairs)")
plt.savefig(f"{OUT_DIR}/fig_value_cooccur_network.png", dpi=150, bbox_inches="tight"); plt.close()
print("Saved fig_value_cooccur_network.png")
